## メイン関数

In [21]:
# --- ここにファイルパスを指定 ---
dat_file = "datas/complex_compare.dat"
solution_file = "results/complex_compare_modified_model (1).txt"

# 1. ファイルからデータをパース
print(f"'{dat_file}' からリズム情報を読み込み中...")
bpm, rhythms = parse_dat_file(dat_file)

print(f"'{solution_file}' から運指データを読み込み中...")
fingerings = parse_solution_file(solution_file)

# 2. データが正しく読み込めた場合のみタブ譜を生成
if bpm is not None and rhythms and fingerings:
    print("\n--- 読み込み成功 ---")
    generate_rhythmic_tab(bpm, rhythms, fingerings)
else:
    print("\nエラー: ファイルの読み込みに失敗したため、タブ譜を生成できませんでした。")

'datas/complex_compare.dat' からリズム情報を読み込み中...
INFO: BPM 190 をファイルから読み込みました。
INFO: availableTime を 111 件読み込みました。
'results/complex_compare_modified_model (1).txt' から運指データを読み込み中...

--- 読み込み成功 ---


                      タブ譜 (Tablature)
    Total(m): 111 | Rests: 7 | Playing Notes: 104
--- TABLATURE (BPM: 190) ---
Legend: sN = Slide to fret N
------------------------------------------------------------------------------------

Rhythm: et------et------st--st--st--et------t--sst-st--et------st--st--st--st--
      e|-----------------------------------------------------------------------
      B|10------10------10-------------------------------------11------9---s10-
      G|--------------------12--10--9-------10-9-------------------------------
      D|-------------------------------------------12--11----------11----------
      A|-----------------------------------------------------------------------
      E|-----------------------------------------------------------------------
Finger: 1----

## Import libraries

In [3]:
import re
import math

## .datファイルからリズムを抽出

In [4]:
def parse_dat_file(dat_filepath):
    """
    .dat ファイルから BPM と availableTime をパースします。
    """
    bpm = 120 # デフォルト
    available_time_ms = []
    try:
        with open(dat_filepath, 'r') as f:
            content = f.read()

        # --- 修正箇所 1: BPM抽出の正規表現を変更 ---
        # 元: r'// BPM of the MIDI file: (\d+)' 
        # 新: r'// BPM:\s*(\d+)'  (スペースのゆらぎも許容)
        bpm_match = re.search(r'// BPM:\s*(\d+)', content)
        if bpm_match:
            bpm = int(bpm_match.group(1))
            print(f"INFO: BPM {bpm} をファイルから読み込みました。")
        else:
            print(f"警告: {dat_filepath} から BPM が見つかりません。デフォルトの {bpm} を使用します。")

        # availableTime抽出
        time_match = re.search(r'availableTime = \[(.*?)\];', content, re.DOTALL)
        if time_match:
            time_str = time_match.group(1).strip()
            # 空要素を除外して変換
            available_time_ms = [int(t.strip()) for t in time_str.split(',') if t.strip()]
            print(f"INFO: availableTime を {len(available_time_ms)} 件読み込みました。")

        if not available_time_ms:
            print(f"警告: {dat_filepath} から 'availableTime' が見つかりませんでした。")

    except FileNotFoundError:
        print(f"エラー: {dat_filepath} が見つかりません。")
        return None, None
    except Exception as e:
        print(f"エラー: {dat_filepath} の読み込み中にエラーが発生しました: {e}")
        return None, None

    return bpm, available_time_ms

def parse_solution_file(solution_filepath):
    """
    CPLEXの出力ファイルから運指データをパースします。
    (変更なし)
    """
    notes_to_play = []
    try:
        with open(solution_filepath, 'r') as f:
            lines = f.readlines()
        
        header_found = False
        for line in lines:
            line = line.strip()
            if not line: continue

            if re.match(r'Note\s+String\s+Fret\s+Handposition\s+Finger', line):
                header_found = True
                continue

            if not header_found: continue
            if 'Objective:' in line or 'Cost Breakdown' in line or 'Total Cost' in line: break
            if line.startswith('---'): break 

            parts = re.split(r'\s+', line)
            
            if len(parts) >= 5:
                note_name = parts[0]
                
                if note_name == 'R':
                    notes_to_play.append({
                        'note': 'R',
                        'string': 0, 
                        'fret': 0, 
                        'finger': 'x'
                    })
                    continue

                try:
                    finger_val = parts[4]
                    parsed_finger = 'x' if finger_val.lower() == 'x' else int(finger_val)

                    notes_to_play.append({
                        'note': note_name,
                        'string': int(parts[1]),
                        'fret': int(parts[2]),
                        'finger': parsed_finger
                    })
                except (ValueError, IndexError):
                    pass

        if not notes_to_play:
             print(f"警告: {solution_filepath} からデータを抽出できませんでした。")

    except FileNotFoundError:
        print(f"エラー: {solution_filepath} が見つかりません。")
        return []
    except Exception as e:
        print(f"エラー: {solution_filepath} の読み込み中にエラー: {e}")
        return []
        
    return notes_to_play

## .txtファイルから実行結果を抽出

## 二つのデータを併せてタブ譜を作成

In [5]:
def generate_rhythmic_tab(bpm, available_time_ms, notes_to_play):
    """
    パースされたデータからテキストタブ譜を生成し、
    ノート数(m)から休符(0)を引いた実演奏数(playingnote)を表示します。
    """
    
    # --- 追加箇所: ノート数の集計ロジック ---
    total_m = len(notes_to_play) # データ上の総数 (m)
    # 既存ロジックに合わせて 'R' を休符(0)としてカウント
    rest_count = sum(1 for n in notes_to_play if n.get('note') == 'R') 
    playing_notes = total_m - rest_count # 実演奏数 (playingnote)
    # ------------------------------------

    print("\n\n" + "="*80)
    print("                      タブ譜 (Tablature)")
    print(f"    Total(m): {total_m} | Rests: {rest_count} | Playing Notes: {playing_notes}") # ここで表示
    print("="*80)

    quarter_note_ms = 60000.0 / bpm
    rhythm_definitions = {
        "q.": (1.5, 14), "q": (1.0, 10), "e.": (0.75, 8),
        "et": (2.0/3.0, 8), "e": (0.5, 6), "s.": (0.375, 5),
        "st": (1.0/3.0, 4), "s": (0.25, 4), "sst": (1.0/6.0, 3),
        "t": (0.125, 3) 
    }
    bpm_scaled_rhythms = sorted(
        [(quarter_note_ms * info[0], (symbol, info[1])) for symbol, info in rhythm_definitions.items()],
        key=lambda x: x[0]
    )

    def get_rhythm_symbol(ms_value):
        if ms_value >= 900: return "START", 6 
        closest_match = min(bpm_scaled_rhythms, key=lambda x: abs(x[0] - ms_value))
        ideal_ms, (symbol, width) = closest_match
        if abs(ideal_ms - ms_value) > (ideal_ms * 0.30) and ideal_ms > 0:
             return "?", 4
        return symbol, width

    print(f"--- TABLATURE (BPM: {bpm}) ---")
    print("Legend: sN = Slide to fret N")
    print("-" * 84 + "\n")

    tab_lines = {
        'rhythm': [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: [], 'finger': []
    }
    line_width = 75
    current_line_length = 0
    
    num_notes = min(len(notes_to_play), len(available_time_ms))
    if len(notes_to_play) != len(available_time_ms):
        print(f"警告: データ数不一致 (運指:{len(notes_to_play)}, 時間:{len(available_time_ms)}) -> {num_notes}個のみ表示")

    labels = { 'rhythm': "Rhythm: ", 1: "e|", 2: "B|", 3: "G|", 4: "D|", 5: "A|", 6: "E|", 'finger': "Finger: " }
    label_width = max(len(l) for l in labels.values())
    
    for key in tab_lines:
        tab_lines[key].append(labels[key].rjust(label_width))
    current_line_length = label_width

    for i in range(num_notes):
        note_info = notes_to_play[i]
        ms_val = available_time_ms[i]
        
        is_rest = (note_info['note'] == 'R')
        
        # --- スライド判定ロジック ---
        is_slide = False
        if i > 0 and not is_rest:
            prev_info = notes_to_play[i-1]
            if prev_info['note'] != 'R':
                if (prev_info['string'] == note_info['string'] and 
                    prev_info['finger'] == note_info['finger'] and
                    prev_info['fret'] != note_info['fret']):
                        is_slide = True
        # ---------------------------

        rhythm_symbol, padding_width = get_rhythm_symbol(ms_val)
        
        s_idx = note_info['string']
        
        # フレット表記の生成
        if is_slide:
            f_str = "s" + str(note_info['fret'])
        else:
            f_str = str(note_info['fret'])
            
        h_str = str(note_info['finger'])
        
        if is_rest:
            f_str = ""
            h_str = ""
            if rhythm_symbol != "START":
                rhythm_symbol += "(R)"

        col_width = max(len(rhythm_symbol), len(f_str), len(h_str), padding_width - 1) + 1

        tab_lines['rhythm'].append(rhythm_symbol.ljust(col_width, '-'))
        tab_lines['finger'].append(h_str.ljust(col_width, '-'))

        for string_index in range(1, 7):
            if not is_rest and string_index == s_idx:
                tab_lines[string_index].append(f_str.ljust(col_width, '-'))
            else:
                tab_lines[string_index].append("-" * col_width)

        current_line_length += col_width

        if current_line_length > line_width:
            for key in ['rhythm', 1, 2, 3, 4, 5, 6, 'finger']:
                print("".join(tab_lines[key]))
            print("\n" + ("-" * (line_width + 2)) + "\n")

            tab_lines = { k: [] for k in tab_lines }
            for key in tab_lines:
                tab_lines[key].append(labels[key].rjust(label_width))
            current_line_length = label_width

    if current_line_length > label_width:
        for key in ['rhythm', 1, 2, 3, 4, 5, 6, 'finger']:
            print("".join(tab_lines[key]))